In [1]:
# Cell 0 — Project Imports

import hashlib
import json
import random
from pathlib import Path

import numpy as np
import torch

In [2]:
# Cell 1 — Patient-Level Dataset Split

def split_patient_ids(
    patient_ids: tuple[str, ...],
    validation_fraction: float,
    test_fraction: float,
    generator: torch.Generator,
) -> dict[str, tuple[str, ...]]:
    """동일 환자가 여러 split에 섞이지 않는 patient-level 분할."""
    if len(set(patient_ids)) != len(patient_ids):
        raise ValueError("patient_ids는 중복 없이 전달해야 합니다.")
    if validation_fraction < 0 or test_fraction < 0 or validation_fraction + test_fraction >= 1:
        raise ValueError("Validation과 test fraction의 합은 1 미만이어야 합니다.")
    order = torch.randperm(len(patient_ids), generator=generator).tolist()
    shuffled = [patient_ids[index] for index in order]
    number_of_test = round(len(patient_ids) * test_fraction)
    number_of_validation = round(len(patient_ids) * validation_fraction)
    test_ids = tuple(sorted(shuffled[:number_of_test]))
    validation_ids = tuple(sorted(shuffled[number_of_test:number_of_test + number_of_validation]))
    training_ids = tuple(sorted(shuffled[number_of_test + number_of_validation:]))
    return {"train": training_ids, "validation": validation_ids, "test": test_ids}

patient_ids = tuple(f"Patient_{index:02d}" for index in range(10))
split_generator = torch.Generator().manual_seed(55254)
patient_split = split_patient_ids(patient_ids, 0.2, 0.2, split_generator)
for split_name, split_ids in patient_split.items():
    print(f"{split_name:10s}: {split_ids}")

train     : ('Patient_00', 'Patient_01', 'Patient_04', 'Patient_06', 'Patient_08', 'Patient_09')
validation: ('Patient_02', 'Patient_07')
test      : ('Patient_03', 'Patient_05')


In [3]:
# Cell 2 — Split Leakage Audit

def audit_disjoint_splits(
    split_to_patient_ids: dict[str, tuple[str, ...]],
) -> tuple[bool, dict[tuple[str, str], tuple[str, ...]]]:
    """모든 split pair의 patient ID 교집합 검사."""
    split_names = tuple(split_to_patient_ids)
    overlaps: dict[tuple[str, str], tuple[str, ...]] = {}
    for left_index, left_name in enumerate(split_names):
        for right_name in split_names[left_index + 1:]:
            shared = tuple(sorted(set(split_to_patient_ids[left_name]) & set(split_to_patient_ids[right_name])))
            overlaps[(left_name, right_name)] = shared
    return all(len(shared) == 0 for shared in overlaps.values()), overlaps

split_is_clean, split_overlaps = audit_disjoint_splits(patient_split)
print("Clean patient-level split:", split_is_clean)
for split_pair, shared_ids in split_overlaps.items():
    print(f"{split_pair}: overlap={shared_ids}")

# 의도적으로 같은 환자를 validation에 삽입한 failure example
leaky_split = {**patient_split, "validation": patient_split["validation"] + (patient_split["train"][0],)}
leaky_is_clean, leaky_overlaps = audit_disjoint_splits(leaky_split)
print("Leaky split accepted:", leaky_is_clean)
print("Detected overlaps:", {pair: ids for pair, ids in leaky_overlaps.items() if ids})

Clean patient-level split: True
('train', 'validation'): overlap=()
('train', 'test'): overlap=()
('validation', 'test'): overlap=()
Leaky split accepted: False
Detected overlaps: {('train', 'validation'): ('Patient_00',)}


In [4]:
# Cell 3 — Train-Only Preprocessing Statistics

def estimate_standardization_statistics(
    training_values: torch.Tensor,  # [N_train,V], floating-point
) -> tuple[torch.Tensor, torch.Tensor]:  # Mean [], standard deviation []
    """Training split만 사용한 normalization 통계 추정."""
    if training_values.ndim != 2 or not torch.is_floating_point(training_values):
        raise TypeError("training_values는 [N_train,V] floating-point Tensor여야 합니다.")
    return training_values.mean(), training_values.std(unbiased=False).clamp_min(1e-8)

train_values = torch.tensor([[0.0, 1.0, 2.0], [1.0, 2.0, 3.0]], dtype=torch.float32)  # [N_train=2,V=3]
test_values = torch.tensor([[100.0, 101.0, 102.0]], dtype=torch.float32)  # [N_test=1,V=3]
train_mean, train_std = estimate_standardization_statistics(train_values)
leaky_mean, leaky_std = estimate_standardization_statistics(torch.cat([train_values, test_values], dim=0))
normalized_test = (test_values - train_mean) / train_std  # [N_test,V]
print("Train-only mean/std:", train_mean.item(), train_std.item())
print("Leaky all-data mean/std:", leaky_mean.item(), leaky_std.item())
print("Normalized test with train statistics:", normalized_test.tolist())

Train-only mean/std: 1.5 0.9574270844459534
Leaky all-data mean/std: 34.66666793823242 46.913631439208984
Normalized test with train statistics: [[102.87989807128906, 103.92436218261719, 104.96882629394531]]


In [5]:
# Cell 4 — Reproducible Random Seed Control

def set_experiment_seed(seed: int, deterministic_algorithms: bool = True) -> None:
    """Python·NumPy·PyTorch 난수 상태와 deterministic mode 설정."""
    if seed < 0:
        raise ValueError("seed는 0 이상이어야 합니다.")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(deterministic_algorithms)

set_experiment_seed(55254)
first_draw = (random.random(), float(np.random.rand()), torch.rand(3))
set_experiment_seed(55254)
second_draw = (random.random(), float(np.random.rand()), torch.rand(3))
same_draw = first_draw[0] == second_draw[0] and first_draw[1] == second_draw[1] and torch.equal(first_draw[2], second_draw[2])
print("First draw:", first_draw)
print("Second draw:", second_draw)
print("Reproduced exactly:", same_draw)

First draw: (0.7529403463805487, 0.5490421153267757, tensor([0.5548, 0.6413, 0.9471]))
Second draw: (0.7529403463805487, 0.5490421153267757, tensor([0.5548, 0.6413, 0.9471]))
Reproduced exactly: True


In [6]:
# Cell 5 — Experiment Manifest와 Locked-Test Gate

def canonical_manifest_hash(manifest: dict[str, object]) -> str:
    """정렬된 JSON 표현의 SHA-256 experiment identity 계산."""
    canonical_json = json.dumps(manifest, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical_json.encode("utf-8")).hexdigest()

def may_evaluate_locked_test(
    model_selection_complete: bool,
    protocol_frozen: bool,
    test_previously_opened: bool,
) -> bool:
    """Protocol 동결 후 단 한 번의 locked-test 평가 허용."""
    return model_selection_complete and protocol_frozen and not test_previously_opened

experiment_manifest: dict[str, object] = {
    "method": "oles3d_adaptive",
    "seed": 55254,
    "split": "fold_0",
    "plans_id": "frozen_plan_v1",
    "training_updates": 10000,
    "sampling_policy": "oles3d_adaptive",
    "code_commit": "REPLACE_WITH_GIT_COMMIT",
}
manifest_hash = canonical_manifest_hash(experiment_manifest)
test_gate_before_freeze = may_evaluate_locked_test(True, False, False)
test_gate_after_freeze = may_evaluate_locked_test(True, True, False)
test_gate_after_open = may_evaluate_locked_test(True, True, True)
print("Manifest JSON:", json.dumps(experiment_manifest, indent=2, sort_keys=True))
print("Manifest SHA-256:", manifest_hash)
print("Before protocol freeze:", test_gate_before_freeze)
print("After protocol freeze:", test_gate_after_freeze)
print("After test already opened:", test_gate_after_open)

Manifest JSON: {
  "code_commit": "REPLACE_WITH_GIT_COMMIT",
  "method": "oles3d_adaptive",
  "plans_id": "frozen_plan_v1",
  "sampling_policy": "oles3d_adaptive",
  "seed": 55254,
  "split": "fold_0",
  "training_updates": 10000
}
Manifest SHA-256: 3ff3cc615ca7ea2b1f31ce4e2d7e7cc9f404d36001fa64b2e97bf123558122a9
Before protocol freeze: False
After protocol freeze: True
After test already opened: False
